# MRR

Monthly Recurring Revenue computed by the Tidemill engine.

## How MRR is normalised

MRR converts every billing interval to a **monthly** rate:

| Interval | Formula                              |
|----------|--------------------------------------|
| month    | `amount / interval_count`            |
| year     | `amount / (12 * interval_count)`     |
| week     | `amount * 52 / (12 * interval_count)`|
| day      | `amount * 365 / (12 * interval_count)`|

Only active subscriptions contribute to current MRR.
All arithmetic uses integer cents to avoid floating-point drift.

In [1]:
from tidemill import reports
from tidemill.reports.client import TidemillClient

reports.setup()

START, END = "2025-06-01", "2026-03-31"
tm = TidemillClient()

## 1. MRR Snapshot

Current Monthly Recurring Revenue and annualised ARR.

In [2]:
reports.mrr.style_snapshot(reports.mrr.snapshot(tm))

MRR,ARR
$909.33,"$10,911.96"


## 2. MRR Breakdown (movements)

Tidemill classifies every MRR change into a **movement type**:

| Movement      | Meaning                                              |
|---------------|------------------------------------------------------|
| **new**       | First subscription for a customer                    |
| **expansion** | Upgrade or additional subscription                   |
| **contraction** | Downgrade (lower plan/fewer seats)                 |
| **churn**     | Cancellation — customer lost entirely                |
| **reactivation** | Previously churned customer returns               |

The breakdown endpoint sums these over the full period.

In [3]:
reports.mrr.plot_breakdown(reports.mrr.breakdown(tm, START, END))

## 3. Monthly MRR with movement split

Query MRR breakdown per month to show how new, expansion, contraction, and churn compose each month's MRR change.

In [4]:
wf = reports.mrr.waterfall(tm, START, END)
reports.mrr.style_waterfall(wf)

,starting_mrr,new,expansion,reactivation,contraction,churn,net_change,ending_mrr
month,,,,,,,,
2025-06,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00
2025-07,$0.00,$908.33,$0.00,$0.00,$0.00,$0.00,$908.33,$908.33
2025-08,$908.33,$60.00,$59.00,$0.00,$0.00,$-80.00,$39.00,$947.33
2025-09,$947.33,$0.00,$0.00,$0.00,$-59.00,$-60.00,$-119.00,$828.33
2025-10,$828.33,$160.00,$0.00,$20.00,$0.00,$-40.00,$140.00,$968.33
2025-11,$968.33,$0.00,$59.00,$0.00,$-59.00,$-139.00,$-139.00,$829.33
2025-12,$829.33,$160.00,$0.00,$0.00,$0.00,$-20.00,$140.00,$969.33
2026-01,$969.33,$100.00,$0.00,$0.00,$0.00,$-80.00,$20.00,$989.33
2026-02,$989.33,$0.00,$0.00,$0.00,$0.00,$-80.00,$-80.00,$909.33


### 3a. Per-customer movement log

Drill into the individual movements that compose each month's waterfall.
Every row is a single customer's MRR change — **new**, **expansion**,
**reactivation**, **contraction**, or **churn** — with monthly subtotals
that should match the waterfall table above.

In [5]:
ml = reports.mrr.movement_log(tm, START, END)
reports.mrr.style_movement_log(ml)

month,date,customer,customer_id,movement,amount
2025-07,2025-07-01,Active Annual Enterprise #7,cus_ULdlfzB8AGoSVW,new,$207.50
2025-07,2025-07-01,Active Annual Pro #6,cus_ULdloJqJKAgH2t,new,$65.83
2025-07,2025-07-01,Active Monthly Pro #4,cus_ULdlStejDAhcFK,new,$79.00
2025-07,2025-07-01,Active Monthly Pro #5,cus_ULdls67szY8zOs,new,$79.00
2025-07,2025-07-01,Active Starter #1,cus_ULdkfi3Mg9S9q6,new,$20.00
2025-07,2025-07-01,Active Starter #2,cus_ULdk3iPcoI9CLe,new,$20.00
2025-07,2025-07-01,Active Starter Heavy #3,cus_ULdkhc6fTXOggE,new,$20.00
2025-07,2025-07-01,Churned Pro #11,cus_ULdlZ6ctuNsyXQ,new,$79.00
2025-07,2025-07-01,Churned Starter #8,cus_ULdlTvyjAKX3Tv,new,$20.00
2025-07,2025-07-01,Churn→Reactivate Starter #15,cus_ULdmofq7kZTDeC,new,$20.00


In [6]:
reports.mrr.plot_waterfall(wf)

In [7]:
reports.mrr.plot_trend(reports.mrr.trend(tm, START, END))